# 01 - SKAB pump anomaly detection

EDA and baseline models on the **Skoltech Anomaly Benchmark (SKAB)**, a water
circulation pump testbed. The goal is a model that scores pump sensor readings
for anomalies, exported to ONNX for `apps/ml` to serve.

Outline: provenance -> loading -> EDA -> features (base and extended) ->
split -> models -> cross-validation -> held-out test -> condition mapping ->
ONNX export.

## 1. Data provenance

- Source: <https://github.com/waico/SKAB> (Iurii Katser, Vyacheslav Kozitsin,
  Skoltech). CSVs live under `data/` in that repo.
- Testbed: a water circulation loop with a pump, valves and sensors sampled
  about once a second.
- Files: `anomaly-free/anomaly-free.csv` (normal operation only, no label
  columns), `valve1/*.csv`, `valve2/*.csv`, `other/*.csv` (one experiment per
  file, each with `anomaly` and `changepoint` labels).
- Format: `;`-separated, `datetime` index, sensors `Accelerometer1RMS`,
  `Accelerometer2RMS`, `Current`, `Pressure`, `Temperature`, `Thermocouple`,
  `Voltage`, `Volume Flow RateRMS`.
- **Licence: GPL-3.0.** Fine for research; check with the team what GPL means
  for a model *trained* on it before shipping that model in the product.

Domain gap worth remembering: SKAB's sensors (accelerometer RMS in g, current,
voltage, ...) are not the platform's metric catalogue (vibration mm/s, power
kW, flow L/min, ...). This notebook is a baseline, not a drop-in production
model.

## 2. Loading

Downloads the CSVs from GitHub into `ml-research/data/skab/` on first run
(`data/` is git-ignored) and reads them from there afterwards. Paths resolve
relative to the notebook, so it works from `uv run jupyter lab`.

In [ ]:
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "skab"
ARTIFACT_DIR = ROOT / "artifacts" / "skab-pump-anomaly"
RAW_URL = "https://raw.githubusercontent.com/waico/SKAB/master/data"

SENSORS = [
    "Accelerometer1RMS",
    "Accelerometer2RMS",
    "Current",
    "Pressure",
    "Temperature",
    "Thermocouple",
    "Voltage",
    "Volume Flow RateRMS",
]

# The repo's data/ tree, verified against the GitHub listing.
FILES = (
    ["anomaly-free/anomaly-free.csv"]
    + [f"valve1/{i}.csv" for i in range(16)]
    + [f"valve2/{i}.csv" for i in range(4)]
    + [f"other/{i}.csv" for i in range(1, 15)]
)
SEED = 42

In [ ]:
def download(files: list[str]) -> None:
    for rel in files:
        target = DATA_DIR / rel
        if target.exists():
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_URL}/{rel}", target)


def load(files: list[str]) -> pd.DataFrame:
    frames = []
    for rel in files:
        df = pd.read_csv(
            DATA_DIR / rel, sep=";", index_col="datetime", parse_dates=True
        )
        # anomaly-free.csv has no label columns: every row is normal.
        for label in ("anomaly", "changepoint"):
            if label not in df:
                df[label] = 0.0
        df["experiment"] = rel
        df["group"] = rel.split("/")[0]
        frames.append(df)
    return pd.concat(frames)


download(FILES)
raw = load(FILES)
raw["anomaly"] = raw["anomaly"].astype(int)
raw["changepoint"] = raw["changepoint"].astype(int)
raw.head()

## 3. EDA

Shape, types, gaps, how the experiments differ, and how the sensors behave
when the pump is anomalous.

In [ ]:
print(raw.shape)
print(raw.dtypes)
raw.isna().sum()

In [ ]:
per_experiment = raw.groupby("experiment").agg(
    group=("group", "first"),
    rows=("anomaly", "size"),
    anomaly_rate=("anomaly", "mean"),
    start=("anomaly", lambda s: s.index.min()),
    end=("anomaly", lambda s: s.index.max()),
)
per_experiment.sort_values(["group", "rows"])

In [ ]:
print(raw["anomaly"].value_counts(normalize=True).rename("share"))
raw.groupby("group")["anomaly"].mean().rename("anomaly_rate")

**Sampling interval.** The models below use row-based rolling windows, which
only mean "N seconds" if the interval is regular.

In [ ]:
deltas = (
    raw.groupby("experiment", group_keys=False)
    .apply(lambda df: df.index.to_series().diff().dt.total_seconds())
    .dropna()
)
deltas.describe()

**One experiment over time**, anomalous stretches shaded red.

In [ ]:
example = raw[raw["experiment"] == "valve1/0.csv"]

fig, axes = plt.subplots(len(SENSORS), 1, figsize=(14, 2 * len(SENSORS)), sharex=True)
for ax, sensor in zip(axes, SENSORS, strict=True):
    ax.plot(example.index, example[sensor], linewidth=0.8)
    ax.fill_between(
        example.index,
        0,
        1,
        where=example["anomaly"] == 1,
        transform=ax.get_xaxis_transform(),
        color="red",
        alpha=0.2,
    )
    ax.set_ylabel(sensor, rotation=0, ha="right")
axes[0].set_title("valve1/0.csv (red = anomaly)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, sensor in zip(axes.flat, SENSORS, strict=True):
    sns.kdeplot(
        data=raw, x=sensor, hue="anomaly", common_norm=False, ax=ax, fill=True
    )
    ax.set_title(sensor)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(raw[SENSORS].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Sensor correlation")
plt.show()

## 4. Feature engineering

A single reading says little; a short window says more. Two feature sets,
both **causal** (past rows only, which is what a live device can compute) and
computed **within each experiment file** so nothing spans two files.

- **base** - per sensor: the raw value plus rolling mean and std over the
  last `WINDOW` (10) rows.
- **extended** - base, plus per sensor:
  - **deviation from the pump's own baseline**: the reading minus the mean of
    the file's first `BASELINE_ROWS` rows, divided by their std (a z-score).
    Each pump/run has its own normal level; this says how far it has drifted
    from it.
  - a longer 60-row rolling mean and std (slow drifts);
  - the rate of change: difference from 10 rows ago.

**How long is the opening normal stretch?** The baseline only makes sense if
those rows are normal. SKAB's own protocol uses the first ~400 rows; check
where the first anomaly actually starts in each file.

In [ ]:
first_anomaly = raw.groupby("experiment")["anomaly"].apply(
    lambda s: int(s.to_numpy().argmax()) if s.any() else len(s)
)
first_anomaly.sort_values().head(5)

Most files stay normal for ~560 rows, but `other/2.csv` turns anomalous at
row 104 (and `other/13.csv` at 495). So `BASELINE_ROWS = 90`: below the
minimum in every file, at the cost of a noisier baseline than SKAB's 400.

**Rows inside the baseline window are dropped** - from every file, including
`anomaly-free.csv`, and for both feature sets, so train, CV and test all use
the same rule and base vs extended are compared on identical rows. A real
device can't score before its baseline exists either. Dropping them also
means every 60-row window and 10-row difference is complete. (The base
numbers therefore shift slightly from earlier runs of this notebook.)

> **Serving requirement (later, not built now):** the extended features need
> a stored **per-device baseline** - the mean and std of the first
> `BASELINE_ROWS` readings after install or maintenance. Production can't
> score a device until it has collected and saved those.

In [ ]:
WINDOW = 10  # rows, about 10 s at SKAB's ~1 Hz
LONG_WINDOW = 60
BASELINE_ROWS = 90


def rolling(by_file, window: int, stat: str) -> pd.DataFrame:
    return by_file.transform(
        lambda s: getattr(s.rolling(window, min_periods=1), stat)()
    )


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    # Reset the datetime index: timestamps repeat across files.
    out = df.reset_index()
    by_file = out.groupby("experiment")[SENSORS]
    row = out.groupby("experiment").cumcount()

    opening = out[SENSORS].where(row < BASELINE_ROWS).groupby(out["experiment"])
    base_mean = opening.transform("mean")
    base_std = opening.transform("std").replace(0, 1)

    out = pd.concat(
        [
            out,
            rolling(by_file, WINDOW, "mean").add_suffix("_mean"),
            rolling(by_file, WINDOW, "std").fillna(0).add_suffix("_std"),
            ((out[SENSORS] - base_mean) / base_std).add_suffix("_z"),
            rolling(by_file, LONG_WINDOW, "mean").add_suffix("_mean60"),
            rolling(by_file, LONG_WINDOW, "std").add_suffix("_std60"),
            by_file.diff(WINDOW).add_suffix("_diff10"),
        ],
        axis=1,
    )
    # No baseline yet inside the window, so those rows can't be scored.
    return out[row >= BASELINE_ROWS].reset_index(drop=True)


data = add_features(raw)
BASE = SENSORS + [f"{s}_mean" for s in SENSORS] + [f"{s}_std" for s in SENSORS]
EXTENDED = BASE + [
    f"{s}{suffix}" for suffix in ("_z", "_mean60", "_std60", "_diff10") for s in SENSORS
]
FEATURE_SETS = {"base": BASE, "extended": EXTENDED}
print(len(BASE), "base features,", len(EXTENDED), "extended features")
assert data[EXTENDED].notna().all().all()
data[EXTENDED].describe().T

## 5. Train / test split by experiment

Random rows would leak: neighbouring seconds of the same run are nearly
identical. Instead a quarter of the files in each group are held out whole.
`anomaly-free.csv` always stays in training - it is the normal reference the
unsupervised models learn from.

The held-out files are touched **once**, at the end (section 7). Everything
that needs tuning - here, the RandomForest's decision threshold - is chosen
from cross-validation on the training files only (section 6).

In [ ]:
rng = np.random.default_rng(SEED)
labelled = [f for f in FILES if not f.startswith("anomaly-free")]
test_files = []
for group in ("valve1", "valve2", "other"):
    members = [f for f in labelled if f.startswith(group)]
    n_test = max(1, len(members) // 4)
    test_files += [str(f) for f in rng.choice(members, size=n_test, replace=False)]

is_test = data["experiment"].isin(test_files)
train, test = data[~is_test], data[is_test]
y_train, y_test = train["anomaly"], test["anomaly"]

print("test files:", sorted(test_files))
print("train rows:", len(train), "anomaly rate:", round(y_train.mean(), 3))
print("test rows: ", len(test), "anomaly rate:", round(y_test.mean(), 3))

## 6. Models

Each model produces a score where higher means more anomalous.

- **Supervised - RandomForest** and **HistGradientBoosting** on all labelled
  training rows, both with balanced class weights, each tried on the base and
  extended features. Trees split on thresholds, so neither needs a scaler,
  and leaving it out keeps the ONNX export exact (see 9).
- **Unsupervised - IsolationForest** and **PCA T²+Q**, base features only,
  kept as reference points. Both are fitted on *normal* training rows only.
  - IsolationForest threshold: 95th percentile of its score on normal rows.
  - PCA T²+Q: standardise, keep the components explaining 95% of the
    variance, then Hotelling **T²** (unusual *inside* the normal subspace)
    and **Q / SPE** (reconstruction error - *outside* it), each thresholded
    at its 99th percentile on normal rows. Score `max(T²/T²_limit,
    Q/Q_limit)`, so 1.0 is the boundary. This is the leaderboard's
    "T-squared+Q", and the relevant family for **real condo pumps, which
    won't come with fault labels**.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    IsolationForest,
    RandomForestClassifier,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_iso() -> Pipeline:
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", IsolationForest(n_estimators=200, random_state=SEED)),
        ]
    )


def make_rf() -> Pipeline:
    return Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=12,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    )


def make_hgb() -> Pipeline:
    return Pipeline(
        [
            (
                "model",
                HistGradientBoostingClassifier(
                    class_weight="balanced", random_state=SEED
                ),
            ),
        ]
    )


class PcaT2Q:
    """PCA monitoring: Hotelling T² + Q (SPE), fitted on normal rows."""

    def __init__(self, variance: float = 0.95, percentile: float = 99):
        self.variance = variance
        self.percentile = percentile

    def fit(self, X: pd.DataFrame) -> "PcaT2Q":
        self.scaler = StandardScaler().fit(X)
        self.pca = PCA(n_components=self.variance, svd_solver="full")
        self.pca.fit(self.scaler.transform(X))
        t2, q = self.statistics(X)
        self.t2_limit = np.percentile(t2, self.percentile)
        self.q_limit = np.percentile(q, self.percentile)
        return self

    def statistics(self, X: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        z = self.scaler.transform(X)
        t = self.pca.transform(z)
        t2 = (t**2 / self.pca.explained_variance_).sum(axis=1)
        q = ((z - self.pca.inverse_transform(t)) ** 2).sum(axis=1)
        return t2, q

    def score(self, X: pd.DataFrame) -> np.ndarray:
        t2, q = self.statistics(X)
        return np.maximum(t2 / self.t2_limit, q / self.q_limit)


def fit_and_score(kind: str, X_fit, y_fit, X_eval):
    """Fit one model; return it, its scores on X_eval and default labels."""
    if kind in ("rf", "hgb"):
        model = (make_rf() if kind == "rf" else make_hgb()).fit(X_fit, y_fit)
        scores = model.predict_proba(X_eval)[:, 1]
        return model, scores, (scores >= 0.5).astype(int)
    normal = X_fit[y_fit == 0]
    if kind == "iso":
        model = make_iso().fit(normal)
        # score_samples is high for normal, so negate it.
        limit = np.percentile(-model.score_samples(normal), 95)
        scores = -model.score_samples(X_eval)
        return model, scores, (scores > limit).astype(int)
    model = PcaT2Q().fit(normal)
    scores = model.score(X_eval)
    return model, scores, (scores > 1).astype(int)


def evaluate(y_true, y_pred, scores) -> dict[str, float]:
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, scores),
    }


# name -> (model kind, feature set)
CANDIDATES = {
    "IsolationForest / base": ("iso", "base"),
    "PCA T²+Q / base": ("pca", "base"),
    "RandomForest / base": ("rf", "base"),
    "RandomForest / extended": ("rf", "extended"),
    "HistGB / base": ("hgb", "base"),
    "HistGB / extended": ("hgb", "extended"),
}
SUPERVISED = [n for n, (kind, _) in CANDIDATES.items() if kind in ("rf", "hgb")]

### 6.1 Grouped cross-validation on the training files

One split of 8 files is noisy - per-file recall swings from near 0 to 0.9.
So before the held-out test, 5-fold `GroupKFold` over the **labelled training
files** (whole files per fold, never rows). `anomaly-free.csv` is added to the
training side of every fold. Supervised models are scored at the default 0.5
here.

The winner is picked on **CV F1 only**, then its threshold is tuned on the
same out-of-fold scores. The held-out split stays as the final test because
of that tuning: CV numbers at the tuned threshold are slightly optimistic,
and the held-out files never influence any choice.

In [ ]:
cv_files = [f for f in labelled if f not in test_files]
cv_data = train[train["experiment"].isin(cv_files)].reset_index(drop=True)
free = train[~train["experiment"].isin(cv_files)]  # anomaly-free.csv

fold_rows = []
oof = pd.DataFrame(
    {"experiment": cv_data["experiment"], "anomaly": cv_data["anomaly"]}
)
for fold, (fit_idx, eval_idx) in enumerate(
    GroupKFold(n_splits=5).split(cv_data, groups=cv_data["experiment"])
):
    fit = pd.concat([free, cv_data.iloc[fit_idx]])
    held = cv_data.iloc[eval_idx]
    for name, (kind, feature_set) in CANDIDATES.items():
        cols = FEATURE_SETS[feature_set]
        _, scores, pred = fit_and_score(kind, fit[cols], fit["anomaly"], held[cols])
        oof.loc[held.index, f"{name} score"] = scores
        oof.loc[held.index, f"{name} pred"] = pred
        fold_rows.append(
            {"fold": fold, "model": name} | evaluate(held["anomaly"], pred, scores)
        )

cv_results = pd.DataFrame(fold_rows)
cv_summary = cv_results.groupby("model", sort=False)[
    ["precision", "recall", "f1", "roc_auc"]
].agg(["mean", "std"])
cv_summary.round(3)

**Winner and its threshold.** The supervised candidate with the best mean CV
F1 wins. A probability of 0.5 is arbitrary, so its threshold is the one that
maximises F1 on its out-of-fold scores - the held-out files play no part.

In [ ]:
WINNER = cv_summary.loc[SUPERVISED, ("f1", "mean")].idxmax()
WIN_KIND, WIN_SET = CANDIDATES[WINNER]
WIN_FEATURES = FEATURE_SETS[WIN_SET]

precision, recall, thresholds = precision_recall_curve(
    oof["anomaly"], oof[f"{WINNER} score"]
)
f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
best = np.argmax(f1[:-1])  # the last P/R point has no threshold
THRESHOLD = round(float(thresholds[best]), 2)
print(f"winner: {WINNER}")
print(f"chosen threshold: {THRESHOLD}  (OOF F1 {f1[best]:.3f})")

oof[f"{WINNER} @ {THRESHOLD} pred"] = (
    oof[f"{WINNER} score"] >= THRESHOLD
).astype(int)

**Per-file recall** from the out-of-fold predictions: which experiments each
candidate actually catches (default thresholds, plus the winner at its tuned
threshold in the last column).

In [ ]:
pred_cols = [c for c in oof.columns if c.endswith(" pred")]
per_file_recall = oof[oof["anomaly"] == 1].groupby("experiment")[pred_cols].mean()
per_file_recall.columns = [c.removesuffix(" pred") for c in pred_cols]
per_file_recall.round(2)

## 7. Held-out test

Refit every candidate on the full training set and score the held-out files
once. Only the **winner** (at its tuned threshold) is the result; the other
rows are for reference, at their default thresholds, and played no part in
any choice.

In [ ]:
results, test_scores = {}, {}
for name, (kind, feature_set) in CANDIDATES.items():
    cols = FEATURE_SETS[feature_set]
    fitted, scores, pred = fit_and_score(kind, train[cols], y_train, test[cols])
    if name == WINNER:
        model = fitted
        pred = (scores >= THRESHOLD).astype(int)
        win_scores, win_pred = scores, pred
        name = f"{name} @ {THRESHOLD} (WINNER)"
    else:
        name = f"{name} (reference)"
    test_scores[name] = scores
    results[name] = evaluate(y_test, pred, scores)

pd.DataFrame(results).T.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, scores in test_scores.items():
    PrecisionRecallDisplay.from_predictions(y_test, scores, name=name, ax=axes[0])
axes[0].set_title("Precision-recall (held-out experiments)")
ConfusionMatrixDisplay.from_predictions(y_test, win_pred, ax=axes[1], colorbar=False)
axes[1].set_title(f"{WINNER} @ {THRESHOLD}")
plt.tight_layout()
plt.show()

**Which to ship.** The **RandomForest on base features**, because it has the
best CV F1: 0.812 ± 0.081, against 0.803 ± 0.097 for HistGB/extended,
0.794 ± 0.087 for HistGB/base and 0.779 ± 0.095 for RF/extended. These gaps
are well inside one standard deviation, so the four are effectively tied.
Its OOF-tuned threshold is 0.49, close to the default. On the held-out files
it scores F1 0.733 (precision 0.94, recall 0.60, ROC-AUC 0.82).

What the extended features change:
- ROC-AUC goes up for both models (RF 0.86 -> 0.91, HistGB 0.85 -> 0.88), and
  precision rises.
- The baseline z-scores help HistGB on `other/12` (recall 0.08 -> 0.53) but
  hurt on `other/4`, `other/11` and `other/14`.
- None of the four catches `other/1`, `other/10`, `other/13` or `valve1/0`.

**On the held-out files, RF/extended looks better** (F1 0.825 vs 0.733).
This notebook doesn't switch to it: the test set is 8 files, and choosing by
test score would make that number optimistic. If extended features are wanted,
the case should come from more CV evidence (e.g. repeated GroupKFold), and
production would then need the per-device baseline (section 4).

The unsupervised models stay far behind (CV F1 0.25 IsolationForest, 0.32
PCA T²+Q). They are fitted on pooled normal rows, not per file as in SKAB's
protocol, which is the direction to explore for unlabelled condo pumps.

**ONNX caveat for HistGB:** with the `skl2onnx` in this environment,
converting `HistGradientBoostingClassifier` fails (`TreeEnsembleClassifier`
rejects a boolean attribute). If HistGB ever wins, that needs an
`skl2onnx`/`onnx` version fix before it can be exported.

Re-read the tables above before trusting this choice on a re-run.

## 8. Scores to asset conditions

The platform's assets are **Good / Needs attention / Faulty**. A simple
mapping on the winner's anomaly probability. "Needs attention" starts at the
tuned decision threshold, so *Needs attention or worse* is exactly the
model's anomaly prediction; "Faulty" is reserved for confident scores. The
Faulty cut-off is a starting point to tune with whoever owns the maintenance
workflow.

In [ ]:
NEEDS_ATTENTION_AT = THRESHOLD
FAULTY_AT = max(0.7, THRESHOLD)


def condition(score: float) -> str:
    if score >= FAULTY_AT:
        return "Faulty"
    if score >= NEEDS_ATTENTION_AT:
        return "Needs attention"
    return "Good"


conditions = pd.Series(win_scores, index=test.index).map(condition)
assert ((conditions != "Good").astype(int) == win_pred).all()
pd.crosstab(conditions, y_test.rename("anomaly"), normalize="columns").round(3)

## 9. Export to ONNX

The winning `Pipeline` goes through `skl2onnx` with a single `float32` input
of shape `[None, n_features]` - what `apps/ml`'s `/predict` reads from the
session signature. `zipmap=False` makes the probabilities a plain tensor
instead of a list of dicts, so they serialise to JSON.

In [ ]:
import onnxruntime as ort
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

onx = to_onnx(
    model,
    initial_types=[("input", FloatTensorType([None, len(WIN_FEATURES)]))],
    options={id(model.named_steps["model"]): {"zipmap": False}},
    target_opset={"": 17, "ai.onnx.ml": 3},
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "model.onnx"
model_path.write_bytes(onx.SerializeToString())

session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
print([(i.name, i.type, i.shape) for i in session.get_inputs()])
print([(o.name, o.type, o.shape) for o in session.get_outputs()])

**Check the ONNX model agrees with sklearn.** ONNX runs on float32, and both
forests and sklearn's gradient boosting compare float32-representable split
thresholds, so the two should match to rounding. A scaler in front breaks
this: ONNX's `Scaler` op rounds differently from sklearn, nudging values
across split thresholds (a 0.07 probability gap when this pipeline had one).

In [ ]:
sample = test[WIN_FEATURES].sample(2000, random_state=SEED).astype(np.float32)
onnx_labels, onnx_proba = session.run(None, {"input": sample.to_numpy()})
sk_proba = model.predict_proba(sample)

max_diff = np.abs(onnx_proba - sk_proba).max()
label_agreement = (onnx_labels == model.predict(sample)).mean()
print(f"max |proba diff|: {max_diff:.2e}, label agreement: {label_agreement:.4f}")
assert max_diff < 1e-3, "ONNX output diverges from sklearn"

**Record how the model was produced** - a metrics file and a short README next
to `model.onnx`, per this folder's conventions. Feature order matters: callers
must send columns in exactly this order.

In [ ]:
win_cv = cv_summary.loc[WINNER]
win_test = results[f"{WINNER} @ {THRESHOLD} (WINNER)"]
estimator = model.named_steps["model"]

metrics = {
    "dataset": "SKAB (https://github.com/waico/SKAB), GPL-3.0",
    "model": type(estimator).__name__,
    "candidate": WINNER,
    "params": estimator.get_params(),
    "feature_set": WIN_SET,
    "n_features": len(WIN_FEATURES),
    "feature_order": WIN_FEATURES,
    "window_rows": WINDOW,
    "long_window_rows": LONG_WINDOW,
    "baseline_rows": BASELINE_ROWS,
    "seed": SEED,
    "test_files": sorted(test_files),
    "onnx_input": {"name": "input", "type": "float32", "shape": [None, len(WIN_FEATURES)]},
    "onnx_outputs": ["label (int64)", "probabilities (float32, [n, 2]; column 1 = anomaly)"],
    # The ONNX `label` output uses 0.5; callers should compare column 1 of
    # `probabilities` against this threshold instead.
    "decision_threshold": THRESHOLD,
    "decision_threshold_source": "best F1 on 5-fold GroupKFold out-of-fold predictions (training files only)",
    "condition_thresholds": {"needs_attention": NEEDS_ATTENTION_AT, "faulty": FAULTY_AT},
    "metrics_cv": {
        name: {m: round(float(v), 4) for m, v in row.items()}
        for name, row in cv_results.groupby("model")[["precision", "recall", "f1", "roc_auc"]].mean().iterrows()
    },
    "metrics_test": {k: {m: round(float(v), 4) for m, v in r.items()} for k, r in results.items()},
}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))

baseline_note = (
    f"""- Needs a per-device baseline: the `_z` features divide by the mean/std
  of the device's first {BASELINE_ROWS} readings (after install or
  maintenance). Nothing can be scored until that baseline is stored.
"""
    if WIN_SET == "extended"
    else ""
)
readme = f"""# skab-pump-anomaly

Produced by `notebooks/01-skab-pump-anomaly.ipynb`.

- Dataset: SKAB, https://github.com/waico/SKAB (GPL-3.0 - check before shipping).
- Model: {type(estimator).__name__} on the `{WIN_SET}` feature set, seed={SEED}.
  Full params in `metrics.json`.
- Features: {len(WIN_FEATURES)} float32 columns, all computed per device from
  past readings only. Exact order in `metrics.json` -> `feature_order`.
{baseline_note}- Input: `input`, float32 `[None, {len(WIN_FEATURES)}]`.
  Outputs: `label` (int64), `probabilities` (float32 `[n, 2]`, column 1 = anomaly).
- Decision threshold: {THRESHOLD} on the anomaly probability (chosen by
  best F1 on out-of-fold CV predictions). The ONNX `label` output uses 0.5 -
  compare `probabilities[:, 1]` against the threshold instead.
- CV (5-fold over training files, at 0.5): F1 {win_cv[("f1", "mean")]:.3f}
  ± {win_cv[("f1", "std")]:.3f}, ROC-AUC {win_cv[("roc_auc", "mean")]:.3f}.
- Held-out experiments: {", ".join(sorted(test_files))}.
- Test metrics (at the threshold): precision {win_test["precision"]:.3f},
  recall {win_test["recall"]:.3f}, F1 {win_test["f1"]:.3f}, ROC-AUC {win_test["roc_auc"]:.3f}.
- Conditions: Good < {NEEDS_ATTENTION_AT} <= Needs attention < {FAULTY_AT} <= Faulty.
"""
(ARTIFACT_DIR / "README.md").write_text(readme)
print(readme)